In [2]:
import pandas as pd

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

# 1. Carregar o modelo de Embeddings (Multilíngue e focado em buscas/classificação)
# O 'intfloat/multilingual-e5-large' é top 1 em vários rankings
model = SentenceTransformer('intfloat/multilingual-e5-large')

# 2. Suas categorias (mantive a lista que você passou)
categorias_gastos = [
    "Restaurantes", "Fast Food", "Padarias e Confeitarias", "Supermercados",
    "Delivery de Comida", "Bares e Lanchonetes", "Combustível", "Estacionamento",
    "Pedágio", "Transporte por Aplicativo (Uber/99)", "Táxi", "Ônibus / Metrô",
    "Manutenção Veicular", "Seguro de Veículo", "IPVA", "Aluguel", "Condomínio",
    "Energia Elétrica", "Água e Esgoto", "Gás", "Internet", "Telefone Fixo",
    "TV por Assinatura", "Materiais de Limpeza", "Manutenção / Reformas",
    "Plano de Saúde", "Farmácias / Medicamentos / Drogaria", "Consultas Médicas",
    "Exames Laboratoriais", "Dentista", "Psicólogo / Terapia", "Academia / Fitness",
    "Ótica", "Mensalidade Escolar", "Mensalidade Universitária", "Cursos Online",
    "Livros e Material Didático", "Papelaria", "Roupas", "Calçados", "Acessórios",
    "Cinema / Teatro", "Shows e Eventos", "Viagens e Hospedagem", "Passagens Aéreas",
    "Streaming (Netflix, Spotify...)", "Jogos e Aplicativos", "Hobbies",
    "Eletrônicos", "Celular / Plano Móvel", "Softwares e Assinaturas",
    "Equipamentos de Informática", "Salão de Beleza / Barbearia",
    "Cosméticos e Perfumaria", "Higiene Pessoal", "Alimentação Animal",
    "Veterinário", "Pet Shop", "Seguros", "Investimentos",
    "Empréstimos / Financiamentos", "Taxas e Tarifas Bancárias",
    "Impostos (IPTU, IR...)", "E-commerce (Amazon, Mercado Livre...)",
    "Assinaturas de Clube", "Doações", "Presentes", "Serviços Domésticos",
    "Cartório e Documentos", "Multas", "Despesas Diversas"
]

# 3. Pré-processamento: O modelo E5 exige o prefixo "passage: " para os itens de busca
# e "query: " para a entrada do usuário.
categorias_formatadas = [f"passage: {c}" for c in categorias_gastos]
cat_embeddings = model.encode(categorias_formatadas, convert_to_tensor=True)

def classificar_fatura(descricao_fatura):
    # Formata a entrada (query)
    query = f"query: {descricao_fatura}"
    query_embedding = model.encode(query, convert_to_tensor=True)
    
    # Calcula a similaridade de cosseno entre a fatura e todas as categorias
    cos_scores = util.cos_sim(query_embedding, cat_embeddings)[0]
    
    # Pega os 3 melhores resultados
    top_results = torch.topk(cos_scores, k=3)
    
    print(f"\nResultado para: '{descricao_fatura}'")
    for score, idx in zip(top_results[0], top_results[1]):
        categoria = categorias_gastos[idx]
        confianca = score.item() * 100
        print(f"{descricao_fatura}-> {categoria}: {confianca:.2f}%")


Loading weights: 100%|██████████| 391/391 [00:01<00:00, 332.07it/s]


In [3]:
df = pd.read_csv('../data/datasets/faturas_nubank.csv')
sample_df = df.iloc[1:10]
textos_amostra = sample_df['title'].tolist()

In [ ]:
for text in textos_amostra:
    classificar_fatura(text)

In [6]:
categorias = [
    "Alimentação em restaurantes, lanchonetes e delivery",
    "Compras em supermercados e mercados",
    "Transporte, combustível, Uber, táxi e estacionamento",
    "Moradia, aluguel, condomínio e contas da casa",
    "Saúde, farmácia, drogaria, consultas e exames",
    "Educação, escola, faculdade e cursos",
    "Compras de roupas, calçados e acessórios",
    "Lazer, cinema, shows e entretenimento",
    "Tecnologia, eletrônicos e informática",
    "Beleza, cosméticos e salão",
    "Pets e veterinário",
    "Serviços diversos",
    "Impostos e taxas",
    "Viagens e hospedagem",
    "Assinaturas e streaming",
    "Investimentos e aplicações",
    "Outros"
]

In [ ]:
from transformers import pipeline
#comparacao 
classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 6612.68it/s]


In [ ]:
from sentence_transformers import util
import torch
def classificar_fatura(texto):
    resultado = classifier(
        f"Compra realizada no estabelecimento: {texto}",
        candidate_labels=categorias,
        hypothesis_template="Esta despesa pertence à categoria {}."
    )

    

    return {
        "categoria": resultado["labels"][0],
        "confiança": f"{resultado['scores'][0]:.1%}",
        "completo": [f"{label}: {score:.1%}" for label, score in zip(resultado['labels'], resultado['scores'])]
    }

def classificar_lote(textos):
    resultados = classifier(
            f"Compra realizada no estabelecimento: {textos}",
            candidate_labels=categorias,
            hypothesis_template="Esta despesa pertence à categoria {}.",
            batch_size=32
        )
    return [
        {
            "categoria": resultado["labels"][0],
            "confiança": f"{resultado['scores'][0]:.1%}\n {resultado['scores'][1]:.1%} \n {resultado['scores'][3]:.1%}"
        }
        for resultado in resultados
    ]

d:\Anaconda\envs\tcc_faturas\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0708 21:16:56.263000 9184 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0708 21:16:56.386000 9184 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


: 

In [5]:
for text in textos_amostra:
    resultado = classificar_fatura(text)
    print(f"{text}-> {resultado['completo']}")

NameError: name 'classificar_fatura' is not defined

In [12]:
classificar_fatura("Drogalis Italo")


Resultado para: 'Drogalis Italo'
Drogalis Italo-> Farmácias / Medicamentos / Drogaria: 82.55%
Drogalis Italo-> Veterinário: 82.53%
Drogalis Italo-> Dentista: 82.03%


In [1]:
def calcula_serie( n ):

    if n <= 1:

        return 1

    x, y = 1, 1

    for i in range(2, n + 1):

        x, y = y, x + y

    return y


n = int(input("Digite o valor de n"))

print(calcula_serie( n ))

21


In [3]:
total = 0
for j in range(1,6):
    total +=j

print(total)

15


In [12]:
s1 = "Bem vindo ao MBA"
s2 = s1
s3 = "Bem vindo ao curso 0"
s4 = "ao"


print (s1 > s3)
print (s2 == s1)
print (s4 < s1)


False
True
False
